# Over-engineering Connect 4

## Motivations

In any field within quantum mechanics, we find that symmetries are often used to make our lives easier and simpler, and they are often unavoidable. Indeed, when Emmy Noether published her theorem in 1918 indicating that our familiar laws are often DEFINED by the symmetries they adhere to, they really are both something to rely on, and something to exploit as often as possible.

Connect 4 is a game where one tries to Connect 4 of the same coloured piece in a grid. It is contested between two players which each have their own colour. Due to it's simple nature, as you may expect, we have already solved the game. Whoever goes first can always force a win, and high level strategies involve just putting as many pieces as you can down the central column and going from there. So you may wonder, how can one overengineer this completed problem to a ridiculous degree, and why were the words 'quantum mechanics' mentioned in the first line. Well we can in fact use quantum machine learning (QML) and exploit the natural symmetries of this game to optimize our solution!

### The setup

Due to budget cuts, I'll be slashing the standard board down to a 4x5 grid, but the problem is scalable. In fact, using geometric QML pays off more when the symmetry group is larger, so feel free to follow along and make the grid as large as you want - sadly my laptop would explode if I went any larger than this.

Let's get the boring coding out the way first:

In [ ]:
import numpy as np
import random

COLUMNS = 5
ROWS = 4

# Initialise an empty board
def create_board():
    return np.zeros((ROWS, COLUMNS))

def possible_moves(board):
    # Return the columns where a piece can be dropped
    moves = []
    rows, cols = board.shape
    for col in range(cols):
        if board[0, col] == 0:
            moves.append(col)
    return moves

def drop_piece(board, col, player):
    # Drop a player's piece into the chosen column
    rows, cols = board.shape
    # Bottom row up to the top
    for row in range(rows - 1, -1, -1):
        # If the space is free, drop the piece
        if board[row, col] == 0:
            board[row, col] = player
            return board
        
def four_in_a_row(board, row, col, d_row, d_col, player):
    # Check for 4 of a player's piece starting at (row, col)
    rows, cols = board.shape
    for step in range(4):
        r = row + step * d_row
        c = col + step * d_col
        # Check for board's boundaries
        if not (0 <= r < rows and 0 <= c < cols):
            return False
        if board[r, c] != player:
            return False
    return True
        
def check_win(board, player):
    # Return True if a 4 in a row anywhere is found
    # Assign directions of right, down, and the diagonals
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

    rows, cols = board.shape
    for row in range(rows):
        for col in range(cols):
            for d_row, d_col in directions:
                if four_in_a_row(board, row, col, d_row, d_col, player):
                    return True
    return False

def random_place(board, player):
    col = random.choice(possible_moves(board))
    return drop_piece(board, col, player)

def evaluate_game(board):
    for player in [-1, 1]:
        if check_win(board, player):
            return player
    if len(possible_moves(board)) == 0:
        return 0 # Draw
    return None

def play_game():
    board, winner = create_board(), None
    while winner is None:
        for player in [1, -1]:
            board = random_place(board, player)
            winner = evaluate_game(board)
            if winner is not None:
                break
    # Flatten the board to use for training
    return [board.flatten(), winner]

A few notes on the game I have devised. Firstly, the game immediately ends when a winner is found. Secondly, the pieces are placed randomly - but this is just to create our dataset that we can learn from. 

## Quantumizing

So we've done the so-called 'classical' part, now to get 'quantum' (silly). 

### Defining our model

I will be defining the problem as follows. Each cell in the connect 4 board will be represented as a qubit. I will label each cell going from the top left, going across the row, then moving to the next row. This makes for 20 qubits, labelled from 0 to 19, all in a line, ready to achieve great things together.

So how does an actual board get loaded onto these qubits? Each cell can hold one of three values, a red piece, a yellow piece, or nothing. Encode these are +1, 0, and -1. Hand each value to its qubit as a rotation, an RX gate turned by 2pi/3. This choice is made as it spaces three possible states evenly around the circuit (0, 2pi/3, -2pi/3), so the circuit sees three cleanly distinguishable inputs instead of three points all bunched together.

But what of the symmetries of the model? Well, looking at our 4x5 board, we can immediately see that we have a point of reflection down the central column. So flipping the board round this axis will yield the same board - or if two players were playing the game, simply turning the board doesn't change the state of the game. Using this idea, we can see where points of symmetry DON'T lie. Let's say one of the players flips the board on its head. Pieces will go everywhere and the game will be ruined, so the existence of gravity actually cuts down our potential symmetries, and we are left with just the one.

There is another symmetry that is not immediately obvious, as it is a different type of symmetry. This would be doing a 'colour-swap' so to say, the yellow pieces to the red pieces. Now, this would of course change the outcome of the game, even though the number of pieces in each column has not changed. This is the idea of equivariance, as opposed to our invariant reflection symmetry. Keep this distinction in mind, as for the mirror we want the model's answer to stay the same, whereas a colour swap should flip the answer along with it. The trick in both cases is to build a circuit that commutes with the symmetry. Let's get to how this can be done.

### Group Theory

To keep it brief, let's only worry about the reflection symmetry. This is a Z2 symmetry: the group has exactly two elements. The first is the identity element, e (do nothing), and the second is the mirror, m (flip down the central column). Mirroring twice lands you right back where you started, so m . m = e, that's the entire group!

The 20 cells of the board are the 'things' the group acts on. The mirror shuffles some cells and leaves the others sitting still. A set of cells that get mapped onto each other by the symmetry is known as an orbit. You can think of these like the cells the group genuinely cannot tell apart, so how many orbits do we have? Using Burnside's lemma, we find that the number of orbits is just the average number of cells left fixed across the group elements:

orbits = (1 / |G|) * sum over g in G of Fix(g)

where Fix(g) counts the cells that g leaves untouched. This group has 2 elements, so |G| = 2, so we just need to average over the two terms. The identify fixes everything, so Fix(e) = 20. The mirror only leaves the central column alone, so Fix(m) = 4. Plugging in, we get orbits = (20 + 4) / 2 = 12 orbits. 

This could have easily been spotted when I defined an orbit, most groups are more complicated than the humble Connect 4 board, but it's nice to be thorough!

## The circuit

Now, the quantum circuit itself. Each qubit will initially be rotated by successive RX and RY gates, but a qubit going off and spinning by itself tells us nothing about the actual game. Taking a step back, Connect 4 is entirely about how pieces relate to one another, so for these qubits to achieve greatness, they have to start communicating with each other. This is done by introducing a 2-qubit gate - the controlled rotation CRY gate. 

It may be tempting to just then entangle every qubit with every other qubit, let's see what that would look like.

Before you think it, let it be known I could have formatted this in a nicer way, but I prefer having the hyperbole of everything being squashed to represent how ridiculous this idea would be. Let's do better. In Connect 4, only winning lines matter. This is the difference between the game being in a playable state or not! The cell in the top left corner can NEVER produce a winning line with the cell in the bottom right corner, so why would we waste precious gates on making them interact? Instead, the CRY gates are only dropped between qubits which could belong to a common winning line. The resulting circuit is by no means simple, but it's certainly better than whatever we had before.

It may initially be hard to see what's going on here, but let me break it down as follows. Firstly, the single qubit gates applied to every qubit are just to set the phase, so let's keep our focus to the 2-qubit gates. Begin with qubit 0, which corresponds to the top left cell on our board. The qubits it shares a 2-qubit gate with are 1, 5, and 6. In other words, winning lines can only be formed with these 3 cells. This makes sense, as a 4 in a row can be made by going horizontally (0, 1, 2, 3), vertically (0, 5, 10, 15), or diagonally (0, 6, 12, 18). Indeed, taking this diagonal line as an example, qubit 6 shares a gate with qubit 12, and qubit 12 shares a gate with qubit 18. Using the simple fact that Connect-4 is defined by 'connecting four', the number of CRY gates has been reduced from 190 to 55.

One important thing to note is that the rules of the game are NOT being hard-coded into the circuit. To put it another way, the circuit doesn't know that 4 in a row means win, we would learn nothing like this. We are pointing the entanglement at the neighbourhoods where wins CAN happen, and the model works out the rest for itself. 

So we know which qubits can talk to each other. The next issue to look at is the fact that every single one of these gates carries its own independent weight, which is an awful lot of dials to tune. In fact, this is the expensive part of any machine learning problem, so how can we do better. Well, good news! All that rambling about symmetries pays itself off here! Flipping the board down the central column gives the same game, so why should two qubits on opposite sides have their own different weights? If they are interchangable, their weights will be tied together so they're always identical.

Let's make that concrete. The single-qubit rotations began as one (RX, RY) per cell, so 20 cells meant 20 separate little dials we will tune. But we already reduced the 20 cells into 12 orbits, so instead of tuning one weight for cell 0 and a completely different one for it's twin, cell 4, just ONE weight is tuned and handed to both. So we end up with 12 dials instead, without throwing anyway anything the symmetry cared about. The entangling gates get the exact same treatment. A CRY doesn't sit on a single cell, but on a pair - a little segment on some potential winning line. Now, mirror this segment, and we get another segment, which is the same kind of line seen from the other side of the board. The game can't tell the two apart, so their weights have no need to be different. The CRY on a line and the CRY on it's mirror image line are tied together into a shared angle. 

Counting this up, the 55 winning-line pairs fold into 29 mirror-orbits, turning 55 indepedent entangling weights into 29 shared ones. 

By forcing the weights to respect the mirror, the entire circuit now commutes with the symmetry. Mirror the board and then run it, or run the board then mirror the result - the final result will be the same. This is a property called equivariance. The parameter count wasn't simply shrunk (even if my humble ThinkPad appreciates it), but the symmetry was literally baked into the model's bones, so it doesn't have to burn training games into something we already know. So we have fewer dials, and the right dials locked in.

### Where is the answer?

A quick note before the code is, where do we actually get the answer from? The system I've described will result in a pile of entangled qubits. That's not a valid solution to connect 4! We need -1, 0, or 1, as I stated earlier. As with everything else, the prediction needs to be invariant under the mirror, so we need to measure something the flip ALSO can't see. This is where the orbits once again pull through for us. For each of the 12 cell-orbits, we will measure the total Pauli-Z over the qubits in that order. A lone central qubit contributes one Z, and a mirror pair contributes the sum of both. Summing over an orbit in this way is how we make sure we measure something the flip cannot detect. Thinking about this the other way, the model literally cannot return a different answer for a board and its reflection, as there is nowhere in the pipeline for that difference to leak through. Hopefully this really solidifies all the hard work we have done with symmetries!

So, what pops out is 12 invariant numbers. We want 3 for our outcomes, so we cap this all off with a plain classical linear layer. This will be a 12 -> 3 matrix plus a bias, and take the largest of the three scores as the call. Nothing quantum about this last step, which is just a consequence of the number of outcomes not matching the number of orbits. Put another way, we actually chose a model fairly light on the symmetries!

### Will this be any good?

A question anyone doing anything should ask themselves. The plan is to have a 1 on 1 deathmatch between two circuits with the same shape and wires, and changing just one thing:

- The equivariant one, weights tied across orbits (12 + 12 + 29 = 53 dials)
- The baseline, where every gate gets its own independent weight (20 + 20 + 55 dials).

Each is then trained on a stingy training set, so just a handful of games, and compare how well each does on test games it has never seen. The whole point of this is to see if baking the symmetry into the model does anything fun!

Word of warning: this group has 2 elements. The rule of thumb is that the generalization benefit scales as the square root of the group size. Sqrt(2) is famously not that big so, don't expect greatness!! I did promise a ridiculously overengineered problem...

Let us commence.

Below is the code to setup the symmetry of our system...

In [6]:
import pennylane as qp
import numpy as np

rows = 4
columns = 5

# Map the coordinates to qubits
# E.g (1, 1) -> 6th qubit
def to_qubit(row, col):
    return row * columns + col

# and the other way...
def to_coord(qubit):
    return divmod(qubit, columns)

# The reflection symmetry
def mirror(row, col):
    # c -> 4 - c (e.g 1st column = 3rd column)
    return row, columns - 1 - col

def cell_orbits():
    seen, orbits = set(), []
    for r in range(rows):
        for c in range(columns):
            q = to_qubit(r, c)
            if q in seen:
                continue
            mq = to_qubit(*mirror(r, c))
            orbit = sorted({q, mq}) # Any mirror pair, or a singleton in the central column
            orbits.append(orbit)
            seen.update(orbit)
    return orbits

# Find which qubit pairs get a CRY
def winning_line_pairs():
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    pairs = []
    for r in range(rows):
        for c in range(columns):
            for dr, dc in directions:
                r2, c2 = r + dr, c + dc
                if 0 <= r2 < rows and 0 <= c2 < columns: # Stay on the board
                    pairs.append((to_qubit(r, c), to_qubit(r2, c2)))
    return pairs

def edge_orbits():
    seen, orbits = set(), []
    for pair in winning_line_pairs():
        key = tuple(sorted(pair))
        if key in seen:
            continue
        (r1, c1), (r2, c2) = to_coord(key[0]), to_coord(key[1])
        mkey = tuple(sorted((to_qubit(*mirror(r1, c1)),
                             to_qubit(*mirror(r2, c2))))) # Mirror both endpoints
        orbit = sorted({key, mkey})
        orbits.append(orbit)
        seen.update({key, mkey})
    return orbits

# 55 CRYs collapse into 29 shared weights 


...and here is the code setting up the circuits. Remember the comparison here is between a circuit that does not respect the symmetry, and one that does.

In [7]:
qubits = 20

dev = qp.device("default.qubit", wires=qubits)

def orbit_observable(orbit):
    obs = qp.PauliZ(orbit[0])
    for q in orbit[1:]:
        obs = obs + qp.PauliZ(q)
    return obs

@qp.qnode(dev)
def circuit(rx_w, ry_w, cry_w, board_flat):
    orbits = cell_orbits()
    edge = edge_orbits()
    observables =  [orbit_observable(o) for o in orbits]

    # Encode the board value as an RX angle, 2pi/3
    for q in range(qubits):
        qp.RX(board_flat[q] * 2 * np.pi /3, wires=q)

    n_layers = rx_w.shape[0]
    for l in range(n_layers):
        # Shared single-qubit rotations. One (rx, ry) per cell orbit
        for k, orbit in enumerate(orbits):
            for q in orbit:
                qp.RX(rx_w[l, k], wires=q)
                qp.RY(ry_w[l, k], wires=q)
        # Shared entanglers. One CRY angle per edge orbit
        for k, orbit in enumerate(edge):
            for (a, b) in orbit:
                qp.CRY(cry_w[l, k], wires=[a, b])

    # Measure the 12 invariant observables
    return [qp.expval(o) for o in observables]

layers = 2
rng = np.random.default_rng(67)
rx_w = rng.normal(size=(layers, len(cell_orbits())))
ry_w = rng.normal(size=(layers, len(cell_orbits())))
cry_w = rng.normal(size=(layers, len(edge_orbits())))

@qp.qnode(dev)
def circuit_no_sym(rx_w, ry_w, cry_w, board_flat):
    orbits = cell_orbits()
    edge = edge_orbits()
    observables =  [orbit_observable(o) for o in orbits]
    # 1. ENCODE — identical
    for q in range(qubits):
        qp.RX(board_flat[q] * 2 * np.pi / 3, wires=q)

    n_layers = rx_w.shape[0]
    for l in range(n_layers):
        # 2. INDEPENDENT single-qubit rotations: one per QUBIT (not per orbit)
        for q in range(qubits):
            qp.RX(rx_w[l, q], wires=q)
            qp.RY(ry_w[l, q], wires=q)
        # 3. INDEPENDENT entanglers: one per winning-line PAIR (not per orbit)
        for k, (a, b) in enumerate(winning_line_pairs()):
            qp.CRY(cry_w[l, k], wires=[a, b])

    # 4. SAME 12 invariant observables
    return [qp.expval(o) for o in observables]

In [9]:
import jax, optax
import jax.numpy as jnp
import random

# Define the winners
LABELS = {1: [1., -1., -1.], 0: [-1., 1., -1.], -1: [-1., -1., 1.]}

def make_dataset(n_games, seed):
    random.seed(seed)
    X, Y = [], []
    for _ in range(n_games):
        board, winner = play_game()
        X.append(np.asarray(board, dtype=float))
        Y.append(LABELS[winner])
    return jnp.array(np.stack(X)), jnp.array(Y)

X, Y = make_dataset(400, seed=67)
print("class counts [P+1, draw, P-1]:", np.bincount(np.argmax(np.array(Y), 1), minlength=3))

def model(params, board, circ):
    rx_w, ry_w, cry_w, W, b = params
    obs = jnp.stack(circ(rx_w, ry_w, cry_w, board))
    return W @ obs + b

def accuracy(params, X, Y, circ, chunk=16):
    correct = 0
    for i in range(0, len(X), chunk):
        preds = jax.vmap(lambda x: model(params, x, circ))(X[i:i+chunk])
        correct += int(jnp.sum(jnp.argmax(preds, 1) == jnp.argmax(Y[i:i+chunk], 1)))
    return correct / len(X)


# The readout W is always 12 -> 3. Only the ansatz shapes differ
def init_params(seed, n_rot, n_cry, layers=2):
    rng = np.random.default_rng(seed)
    return [
        jnp.array(rng.normal(size=(layers, n_rot))), # rx
        jnp.array(rng.normal(size=(layers, n_rot))), # ry
        jnp.array(rng.normal(size=(layers, n_rot))), # cry
        0.1 * jnp.array(rng.normal(size=(3, 12))), # W
        jnp.zeros(3) # b
    ]

def train(circ, params, Xtr, Ytr, Xte, Yte, steps=120, lr=0.05):
    opt = optax.adam(lr)
    state = opt.init(params)

    @jax.jit
    def step(params, state):
        def loss(p):
            preds = jax.vmap(lambda x: model(p, x, circ))(Xtr)
            return jnp.mean((preds - Ytr) ** 2)
        l, g = jax.value_and_grad(loss)(params)
        updates, state = opt.update(g, state)
        return optax.apply_updates(params, updates), state, l
    
    for s in range(steps):
        params, state, l = step(params, state)
        if s % 20 == 0:
            print(f"step {s:3d} | loss {l:.3f} | train {accuracy(params,Xtr,Ytr,circ):.2f} "
                  f"| test {accuracy(params,Xte,Yte,circ):.2f}")
    return params

N_train = 30
Xtr, Ytr = X[:N_train], Y[:N_train]
Xte, Yte = X[N_train:N_train+150], Y[N_train:N_train+150]

orbits = cell_orbits()

print("=== EQUIVARIANT (shared weights) ===")
p_eq = init_params(0, n_rot=len(orbits), n_cry=len(edge_orbits()))   # 12, 29
p_eq = train(circuit, p_eq, Xtr, Ytr, Xte, Yte)

print("\n=== BASELINE (independent weights) ===")
p_bl = init_params(0, n_rot=qubits, n_cry=len(winning_line_pairs()))            # 20, 55
p_bl = train(circuit_no_sym, p_bl, Xtr, Ytr, Xte, Yte)

class counts [P+1, draw, P-1]: [165 106 129]
=== EQUIVARIANT (shared weights) ===
step   0 | loss 0.999 | train 0.67 | test 0.43


JaxRuntimeError: INTERNAL: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Error dispatching computation: Out of memory allocating 48570050500 bytes.